<a href="https://colab.research.google.com/github/Alisherbakh/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install transformers torch bitsandbytes datasets peft

In [34]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch
from datasets import load_dataset

In [35]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'      # Hugging Face'dagi model nomi: "muallif/model_nomi"
tokenizer = AutoTokenizer.from_pretrained(model_id)  # Shu modelga mos tayyor tokenizerni (lug'at + bo'lish qoidalari) yuklab olamiz

#print('Vocab size:', tokenizer.vocab_size)               # Lug'atdagi tokenlar soni → 32000
#print('Special tokens:', tokenizer.special_tokens_map)   # Maxsus tokenlar → {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}


# quantization
bnb_config = BitsAndBytesConfig(                  # Modelni siqish (quantization) sozlamalari
    load_in_4bit=True,                            # Og'irliklarni 4 bitga siqib yuklaymiz (odatda 16 bit bo'ladi)
    bnb_4bit_use_double_quant=True,               # Siqish uchun kerak bo'lgan yordamchi sonlarni ham siqamiz → yana ozgina xotira tejaladi
    bnb_4bit_compute_dtype=torch.bfloat16,        # Hisoblash paytida sonlar vaqtincha bfloat16 ga ochiladi (aniqlik uchun)
    bnb_4bit_quant_type='nf4'                     # Siqish usuli: NF4 — neyron tarmoq og'irliklari uchun maxsus 4 bitli format
)

bnb_config                                        # Sozlamalarni ekranga chiqaramiz (tekshirish uchun)

model = AutoModelForCausalLM.from_pretrained(     # Matn yozuvchi (keyingi tokenni bashorat qiluvchi) tayyor modelni yuklaymiz
    model_id,                                     # Qaysi model (tokenizer bilan bir xil bo'lishi shart)
    quantization_config=bnb_config,               # Yuqoridagi 4 bitli siqish sozlamalarini qo'llaymiz
    device_map='auto',   # GPU, CPU                 → model qismlarini GPU va CPU'ga avtomatik taqsimlaydi
    dtype=torch.bfloat16                          # Siqilmagan qismlar uchun ma'lumot turi
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [36]:
# Before Fine-tuning                                   → modelni o'zimizning ma'lumotda o'qitishdan OLDIN qanday javob berishini ko'ramiz
prompt = "Explain what a tokenizer is?"                # Modelga beriladigan savol (prompt)
# prompt = "A tokenizer is a tool in natural language processing that"   → muqobil: gapni boshlab berib, davomini yozdirish

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)   # GPU, CPU   → promptni tensorga aylantirib, model turgan qurilmaga ko'chiramiz

with torch.no_grad():                                  # Gradient hisoblamaymiz (faqat javob olyapmiz, o'qitmayapmiz)
    output_ids = model.generate(                       # Model javob matnini token-token yozadi
        **inputs,                                      # input_ids va attention_mask ni argument qilib beramiz
        max_new_tokens=80,                             # Ko'pi bilan 80 ta YANGI token yozsin
        do_sample=True,                                # Har safar eng ehtimollisini emas, ehtimollik bo'yicha tasodifiy tanlaydi
        temperature=0.7                                # Tasodifiylik darajasi: past → aniqroq, baland → ijodkorroq
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))   # Javob ID'larini matnga aylantirib chiqaramiz (<s>, </s> siz)

# print(model)                                  → butun modelni chiqarish o'chirib qo'yildi (juda uzun)
first_block = model.model.layers[0]             # 22 ta Transformer blokdan birinchisini (0-indeks) olamiz
print('first_block:', first_block)              # Shu bitta blokning ichki tuzilmasini chiqaramiz
print('=======')                                # Ajratuvchi chiziq
print(first_block.self_attn)                    # Blok ichidagi Attention (diqqat) qismini alohida chiqaramiz
print('=======')                                # Ajratuvchi chiziq
print(model.config)                             # Modelning barcha sozlamalarini (o'lchamlar, qatlamlar soni) chiqaramiz

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is?
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_feat

In [37]:
def count_parameters(model):                                  # Modeldagi parametrlar sonini sanaydigan funksiya
    return sum(p.numel() for p in model.parameters())         # Har bir parametr tensoridagi sonlar miqdorini olib, hammasini qo'shamiz

total_params = count_parameters(model)                        # Funksiyani modelimizga qo'llaymiz
print(f"Total parameters (including frozen 4-bit): {total_params:,}")   # Natijani mingliklarni vergul bilan ajratib chiqaramiz → 615,606,272

Total parameters (including frozen 4-bit): 615,606,272


# dataset library | load_dataset



*  instruction tuning




In [38]:
from datasets import load_dataset                            # Hugging Face'ning tayyor datasetlarni yuklovchi funksiyasi

dataset = load_dataset("yahma/alpaca-cleaned", split="train")   # "alpaca-cleaned" datasetini internetdan yuklaymiz, faqat "train" qismini
dataset
dataset[1]                                                     # Dataset haqida qisqa ma'lumotni ko'rsatamiz

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

In [39]:
def generate_prompt(example):                     # Bitta misoldan (instruction, input, output) bitta prompt matni yasaydigan funksiya
  instruction = example['instruction']            # Topshiriq / savolni olamiz
  input_text = example['input']                   # Qo'shimcha ma'lumotni olamiz (ko'pincha bo'sh "")
  output_text = example['output']                 # To'g'ri javobni olamiz

  if input_text:                                  # Agar input bo'sh bo'lmasa (qo'shimcha ma'lumot bor bo'lsa)...
    return(                                       # ...3 qismli shablon qaytaramiz:
        "### Instruction:\n"                      # Topshiriq sarlavhasi + yangi qator
        f"{instruction}\n\n"                      # Topshiriq matni + bo'sh qator
        "### Input:\n"                            # Input sarlavhasi
        f"{input_text}\n\n"                       # Input matni + bo'sh qator
        "### Response:\n"                         # Javob sarlavhasi
        f"{output_text}"                          # Javob matni
    )
  else:                                           # Agar input bo'sh bo'lsa...
    return(                                       # ...2 qismli shablon qaytaramiz (Input qismisiz):
       "### Instruction:\n"                       # Topshiriq sarlavhasi
       f"{instruction}\n\n"                       # Topshiriq matni
       "### Response:\n"                          # Javob sarlavhasi
       f"{output_text}"                           # Javob matni
    )

# generate_prompt(dataset[0])                     → tekshirish uchun: birinchi misol qanday ko'rinishini ko'rish

def formatting_func(example):                     # map() uchun yordamchi funksiya
  return {'text': generate_prompt(example)}       # Yangi 'text' ustuniga tayyor prompt matnini yozamiz

dataset = dataset.map(formatting_func)            # Funksiyani datasetdagi HAR BIR qatorga qo'llaymiz → yangi 'text' ustuni qo'shiladi

In [40]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [41]:
dataset = dataset.select(range(7000))     # 51 760 ta misoldan faqat birinchi 7000 tasini qoldiramiz (o'qitish tezroq bo'lishi uchun)


In [42]:
dataset = dataset.shuffle(seed=42)        # Qolgan 7000 ta misolni aralashtiramiz; seed=42 → har safar bir xil tartibda aralashadi

In [43]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [ ]:
# Full Fine-tuning  =>                        → modelning BARCHA parametrlarini qayta o'qitish (qimmat, katta xotira)
# Cheap Fine-tuning =>                        → faqat kichik qismini o'qitish (arzon, kam xotira)
# PEFT => Parameter Efficent Fine Tuning      → parametrlardan tejamkor foydalanib fine-tuning qilish usullari (LoRA shu turga kiradi)
# OOM => Out of Memory                        → GPU xotirasi yetmay qolganda chiqadigan xato

In [47]:
from peft import LoraConfig, get_peft_model                   # PEFT kutubxonasidan LoRA sozlamalari klassini chaqiramiz

lora_config = LoraConfig(                          # LoRA qanday qo'shilishini belgilaymiz
    r=8,   # rank                               → qo'shiladigan kichik matritsalarning "kengligi" (A: 2048×8, B: 8×2048)
    lora_alpha=16,                            # LoRA ta'sirining kuchi (masshtab): ta'sir = alpha / r = 16 / 8 = 2
    lora_dropout=0.05,                        # O'qitishda LoRA neyronlarining 5% ini tasodifan o'chiradi (yodlab olmaslik uchun)
    task_type="CAUSAL_LM",                    # Vazifa turi: keyingi tokenni bashorat qiluvchi matn modeli
    target_modules=["q_proj", "v_proj"],      # LoRA faqat Attention'dagi Query va Value qatlamlariga qo'shiladi
)

In [48]:
model = get_peft_model(model, lora_config) # LoRA sozlamalarini modelga qo'llaymiz: q_proj va v_proj yoniga kichik A, B matritsalar qo'shiladi


In [49]:
model.print_trainable_parameters()            # Nechta parametr o'qitilishini va umumiy sondan necha foiz ekanini chiqaramiz

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# QLoRa    → Quantized LoRA: model 4 bitga siqiladi + ustiga LoRA qo'shiladi (siz hozir shuni qilyapsiz)
# LoRa     → Low-Rank Adaptation: asl model muzlatiladi, yoniga kichik A va B matritsalar qo'shilib, faqat ular o'qitiladi